In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.backends.backend_pdf import PdfPages

In [ ]:
# Load CSV file
file_path = "Figure_S1A_TSC22D1_TSC22D2_TSC22D4_AP-MS_PhosphositePlus_Combined.csv"  # Replace with your actual CSV file path
df = pd.read_csv(file_path)

# Define the column groups for each TSC22D
columns_groups = {
    "TSC22D1": ["Residue_TSC22D1", "Phosphosite_TSC22D1", "MSdetected_TSC22D1"],
    "TSC22D2": ["Residue_TSC22D2", "Phosphosite_TSC22D2", "MSdetected_TSC22D2"],
    "TSC22D4": ["Residue_TSC22D4", "Phosphosite_TSC22D4", "MSdetected_TSC22D4"],
}

# Path for the output PDF
output_pdf_path = "Figure_S1A_All_TSC22D_AP-MS_Colormap.pdf"

# Open a PdfPages object
with PdfPages(output_pdf_path) as pdf:
    for key, columns in columns_groups.items():
        try:
            print(f"Processing {key}...")  # Debug output

            # Extract columns
            residues = df[columns[0]]  # Residue column (y-axis)
            phosphosites = df[columns[1]]  # Phosphosite column (x-axis)
            ms_detected = df[columns[2]]  # MSdetected column (for color)

            # Convert phosphosites to numeric, coercing errors to NaN
            phosphosites = pd.to_numeric(phosphosites, errors='coerce')

            # Filter out rows where phosphosites are zero or NaN
            valid_indices = (phosphosites != 0) & (~phosphosites.isna())
            residues = residues[valid_indices]
            phosphosites = phosphosites[valid_indices]
            ms_detected = ms_detected[valid_indices]

            # Ensure matching lengths for all three columns
            min_len = min(len(residues), len(phosphosites), len(ms_detected))
            residues = residues[:min_len]
            phosphosites = phosphosites[:min_len]
            ms_detected = ms_detected[:min_len]

            # Assign colors based on MSdetected values
            colors = ms_detected.map({False: '#00224E', True: '#FEE000'})

            # Create a new figure
            plt.figure(figsize=(0.8, 1.8))

            # Plot lollipop lines
            plt.hlines(y=residues, xmin=0, xmax=phosphosites, color='#E6E6E6', alpha=0.7, linewidth=1)

            # Plot lollipop dots with conditional colors
            plt.scatter(phosphosites, residues, color=colors, s=25, alpha=0.8)

            # Customize the plot
            plt.gca().invert_yaxis()  # Flip the direction of the y-axis
            plt.xticks([])  # Remove x-axis ticks
            plt.yticks([])  # Remove y-axis ticks
            plt.grid(alpha=0.3)
            plt.tight_layout()

            # Save the current plot into the PDF
            pdf.savefig()  # Save the figure
            plt.close()  # Close the figure to free memory

            print(f"{key} processed successfully.")  # Debug output
        except Exception as e:
            print(f"Error processing {key}: {e}")  # Debugging for errors

# Confirmation
print(f"All plots have been saved into a single PDF: {output_pdf_path}")